# PREPROCESSING

We are going to preprocess our corpus that is in obituaries.csv.

In [2]:
# --- Standard Library & System ---
import os
import sys
import subprocess
import re
import warnings
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

# --- NLP & Processing ---
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm

# --- Custom Module (Your own file) ---
from obituary_parser import parse_pairs_nonbatch, parse_pairs_parallel

# --- Configuration ---
warnings.filterwarnings('ignore')
tqdm.pandas()

# --- Functions & Model Loading ---
def install_spacy_model(model_name):
    try:
        return spacy.load(model_name)
    except OSError:
        print(f"Downloading spaCy model: {model_name}")
        subprocess.check_call([sys.executable, "-m", "spacy", "download", model_name])
        return spacy.load(model_name)

# Usas spaCy para todo el proceso de NLP (lematización, stop words, etc.)
sp = install_spacy_model('en_core_web_sm')

print("All imports successful!")

All imports successful!


In [6]:
df = pd.read_csv('obituaries.csv')

**1. Create the gender variable**

To do so, we define a list of words that can be associated with the gener of the person (she, he, her, his...), and based on that, we count the instances of this word in each biography, and determine if it's a men, a women o unknown (if both counts are the same).

DON'T RUN IT IF IT'S NOT STRICTLY NECESSARY! (It takes a while)

In [ ]:
def detect_gender(text):
    if not isinstance(text, str):
        return "Unknown"
    
    # Define the words to "detect" the gender
    masculi = r'\b(he|his|him|himself|mr)\b'
    femeni = r'\b(she|her|hers|herself|mrs|ms|miss)\b'
    
    # Count the instances
    count_m = len(re.findall(masculi, text.lower()))
    count_f = len(re.findall(femeni, text.lower()))
    
    if count_m > count_f:
        return "M"
    elif count_f > count_m:
        return "F"
    else:
        return "Unknown" 

try:
    
    col_text = 'biography'

    df['gender'] = df[col_text].apply(detect_gender)

    # Some statistics of the results
    print("Results of classifying by gender:")
    print(df['gender'].value_counts())

    # Save the new file
    df.to_csv('obituaries_gender.csv', index=False)

except Exception as e:
    print(f"Error: {e}")

We save the results in a new file obituaries_gender.csv, so we don't have to rerun it in the futur (it takes a while).

In [10]:
df = pd.read_csv('obituaries_gender.csv')
df = df[df["gender"]!= "Unknown"]
df = df.reset_index(drop=True)

Also, after doing the next step, we will not take into acount those observations with `gender` unknown, as it's crucial for our study.

**2. Split the biographies in different subsets**

To do so, we use paralelization because otherwise it takes too much time. First we create four new variables from the biography text: biography (part of the orbituary where it talks about birth date, death date...), family (part of the orbituary where is exposed the survivors of the person), memorial (part of the orbituary where it talks about the memorial ritual) and life (everything else). We do that because for our study we only want to analize the life part, that is the history of the person and the way the writter remeber they.

In [11]:
RUN_MODE = "parallel"  # "parallel" or "nonbatch"
CHUNK_SIZE = 1000
MAX_ROWS = None  # set e.g. 200 for quick tests; keep None for full dataset
MAX_WORKERS = max(1, (os.cpu_count() or 1) - 1)

In [12]:
MAX_ROWS = None
source_df = df[['date', 'biography']] if MAX_ROWS is None else df[['date', 'biography']].iloc[:MAX_ROWS]
source_series = source_df.apply(lambda row: (row['date'], row['biography']), axis=1)
pairs = list(source_series.items())

if RUN_MODE == "nonbatch":
    parse_data = parse_pairs_nonbatch(pairs)
else:
    parse_data = parse_pairs_parallel(
        pairs,
        chunk_size=CHUNK_SIZE,
        max_workers=MAX_WORKERS,
    )

Processing Sections: 100%|██████████| 216/216 [00:30<00:00,  7.19it/s]


In [13]:
result = pd.DataFrame.from_dict(parse_data, orient='index').sort_index()
sections_df = result['sections'].apply(pd.Series).replace({'': pd.NA})
df_final = pd.concat([df, sections_df], axis=1)
df_final.to_csv('obituaries_life.csv', index=False)

We have saved the data a little bit more cleaned in the csv file obituaries_life.csv.

In [14]:
df = pd.read_csv('obituaries_life.csv')

**3. Delete proper nouns and numbers**

Checking the date we have seen that there are a lot of proper nouns and numbers in the biography, which are not usefull at all, so we have decided to delete them before doing lemmatization in order to keep only meaningful words.

In [15]:
def remove_names_and_numbers(text):
    if not isinstance(text, str):
        return ""

    text_clean = re.sub(r'[.,()!?]', '', text)

    words = text_clean.split()
    
    result_words = []
    for word in words:
        # Delete all the words that start with a capital letter and all the numbers. 
        if not word[0].isupper() and not any(char.isdigit() for char in word):
            result_words.append(word)
    
    return " ".join(result_words).strip()


print("Cleaning names and numbers by removing words entirely...")
df['life_no_names_numbers'] = df['life'].apply(remove_names_and_numbers)

# Preview
print("\nExample:")
print(f"Original: {df['life'].iloc[0]}...")
print(f"Cleaned:  {df['life_no_names_numbers'].iloc[0]}")

df.to_pickle('obituaries_no_names.pkl')

Cleaning names and numbers by removing words entirely...

Example:
Original: In loving memory of William Richard “Butch” Lynch, 6/14/49 - 12/31/25. Formerly of Roxbury, Butch lived in South Boston for many years. He passed peacefully on December 31, 2025 with family by his side. Butch was the much loved brother of John “Guy” Lynch, Jacqueline “Sissy “ Lynch, Margaret “Margie” Goodine, and the late James Lynch and Joseph Lynch. Butch loved traveling, spending time with family, friends and good food. He was very talented and enjoyed working on home remodeling projects, solving complex mathematical equations, and was always ready to help family, friends and neighbors. Butch was known for his sense of humor, wit and fun, his eclectic taste in music and he loved to sing, especially his Motown favorites. As his nephew Patrick observed, Butch was a source of common sense, logic and good values, or just the right amount of silliness, depending on what the situation needed. Our family would lik

We know that deleting all the words starting with a capial letter we are also deleting the first word of a sentence, but that's not a problem because we have checked that normally they are not meaningful words. 

Finally we have saved the data with the new variable `life_no_names_numbers` with our data cleaned a little bit more than before in a pickle file.

In [16]:
df = pd.read_pickle("obituaries_no_names.pkl")

**4. Apply lemmatization**

We have decided to apply lemmatization to our corpus because we think that is the most useful one for our research. We also run it in paralel for eficiency. We have salved the resulting data frame in a pickle file so that we don't have to rerun it in the futur. 

In [17]:
def parallel_preprocess(texts):

    clean_texts = [str(t) if pd.notnull(t) else "" for t in texts]
    processed_texts = []
    
    for doc in tqdm(sp.pipe(clean_texts, n_process=MAX_WORKERS if RUN_MODE == "parallel" else 1, batch_size=CHUNK_SIZE, disable=['ner', 'parser']), total=len(clean_texts)):
        tokens = [token.lemma_ for token in doc 
                  if not token.is_stop and not token.is_punct and token.lemma_.strip() != '']
        processed_texts.append(" ".join(tokens))
    
    return processed_texts

df["text_clean"] = parallel_preprocess(df['life_no_names_numbers'])

df.to_pickle('obituaries_lemma.pkl')

100%|██████████| 215564/215564 [18:55<00:00, 189.89it/s] 
